# 🎙️ Chatterbox TTS — Colab GPU Server

This notebook runs Chatterbox Multilingual V3 on a free Colab T4 GPU
and exposes it over ngrok so your local VoiceLink backend can call it remotely.

## Quick Start
1. **Runtime → Change runtime type → T4 GPU**
2. **Run all cells** (Runtime → Run all)
3. Copy the **Public URL** printed at the bottom
4. Paste it into your local `.env` as `COLAB_TTS_URL=<url>`

⚠️ The URL changes every time you restart the runtime.

In [ ]:
#@title Cell 1 — Install dependencies (run first)
#@markdown Installs chatterbox-tts WITHOUT downgrading Colab's torch.
!pip install -q chatterbox-tts --no-deps
!pip install -q fastapi uvicorn pyngrok python-multipart soundfile numpy huggingface_hub transformers accelerate

import torch
print(f"✅ torch {torch.__version__}, CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
#@title Cell 2 — Load Chatterbox model on GPU
import torch
from chatterbox.mtl_tts import ChatterboxMultilingualTTS

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

import time as _time
_t0 = _time.time()
model = ChatterboxMultilingualTTS.from_pretrained(device=device)
print(f"✅ Model loaded on {device} ({dtype}) in {_time.time() - _t0:.1f}s")
if device == "cuda":
    print(f"   VRAM: {torch.cuda.memory_allocated() / 1024**2:.0f} MB allocated")

In [ ]:
#@title Cell 3 — Start FastAPI server + ngrok tunnel
# ── Imports ──────────────────────────────────────────────────────────
import base64
import io
import os
import socket
import subprocess
import threading
import time

import numpy as np
import soundfile as sf
import torch
import uvicorn
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
from pyngrok import ngrok

# ── Verify model from Cell 2 is loaded ───────────────────────────────
assert 'model' in dir() and model is not None, \
    "❌ Model not loaded! Run Cell 2 first (Runtime → Run cell)."
assert 'device' in dir(), \
    "❌ 'device' not defined! Run Cell 2 first."
print(f"✅ Using model on {device}")

# ── Supported languages ──────────────────────────────────────────────
SUPPORTED_LANGUAGES = {
    "ar", "da", "de", "el", "en", "es", "fi", "fr", "he", "hi",
    "it", "ja", "ko", "ms", "nl", "no", "pl", "pt", "ru", "sv",
    "sw", "tr", "zh",
}

# ── FastAPI app ──────────────────────────────────────────────────────
app = FastAPI(title="Chatterbox TTS Server")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

class SynthesizeRequest(BaseModel):
    text: str = Field(..., min_length=1, max_length=5000)
    reference_audio_b64: str = Field(..., description="Base64-encoded WAV reference audio")
    language: str = Field("en", description="ISO language code")
    speed: float = Field(1.0, ge=0.5, le=2.0)

@app.get("/health")
def health():
    return {
        "status": "ok",
        "device": device,
        "model_loaded": model is not None,
        "cuda_available": torch.cuda.is_available(),
    }

@app.post("/synthesize")
def synthesize(req: SynthesizeRequest):
    """Synthesize speech with a cloned voice."""
    if req.language not in SUPPORTED_LANGUAGES:
        raise HTTPException(
            status_code=400,
            detail=f"Unsupported language '{req.language}'. Supported: {', '.join(sorted(SUPPORTED_LANGUAGES))}",
        )

    # Decode reference audio
    try:
        ref_bytes = base64.b64decode(req.reference_audio_b64)
        ref_audio, ref_sr = sf.read(io.BytesIO(ref_bytes))
    except Exception as exc:
        raise HTTPException(status_code=400, detail=f"Invalid reference audio: {exc}")

    # Save temp WAV for Chatterbox (it wants a file path)
    tmp_path = "/tmp/ref_tmp.wav"
    sf.write(tmp_path, ref_audio, ref_sr, format="WAV")

    t0 = time.time()
    try:
        wav = model.generate(
            text=req.text,
            language_id=req.language,
            audio_prompt_path=tmp_path,
        )
    except Exception as exc:
        raise HTTPException(status_code=500, detail=f"Model inference failed: {exc}")
    elapsed = time.time() - t0

    # Convert tensor → WAV bytes
    if isinstance(wav, torch.Tensor):
        audio_data = wav.cpu().numpy()
    else:
        audio_data = np.asarray(wav, dtype=np.float32)
    if audio_data.ndim > 1:
        audio_data = audio_data.squeeze()
    audio_data = audio_data.astype(np.float32)
    max_val = np.abs(audio_data).max()
    if max_val > 1.0:
        audio_data = audio_data / max_val

    sr = getattr(model, "sr", 24000)
    buf = io.BytesIO()
    sf.write(buf, audio_data, samplerate=sr, format="WAV")
    audio_b64 = base64.b64encode(buf.getvalue()).decode()

    print(f"[SYNTH] text={req.text[:60]!r} lang={req.language} {elapsed:.2f}s", flush=True)
    return {
        "audio_base64": audio_b64,
        "mime_type": "audio/wav",
        "duration_s": round(len(audio_data) / sr, 2),
        "inference_time_s": round(elapsed, 3),
    }

# ── Kill any process on port 8899 ────────────────────────────────────
PORT = 8899
try:
    _s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    _s.connect(('127.0.0.1', PORT))
    _s.close()
    print(f'⚠️  Port {PORT} in use — killing old process...')
    try:
        subprocess.run(['fuser', '-k', f'{PORT}/tcp'], capture_output=True)
    except FileNotFoundError:
        # fuser not available — use lsof fallback
        result = subprocess.run(['lsof', '-ti', f':{PORT}'], capture_output=True, text=True)
        if result.stdout.strip():
            pid = result.stdout.strip().split('\n')[0]
            subprocess.run(['kill', '-9', pid], capture_output=True)
    time.sleep(1)
except (ConnectionRefusedError, OSError):
    pass  # port is free

# ── Start uvicorn in background ─────────────────────────────────────
def _run_server():
    uvicorn.run(app, host="127.0.0.1", port=PORT, log_level="info")

server_thread = threading.Thread(target=_run_server, daemon=True)
server_thread.start()
time.sleep(2)  # let the server bind
print(f"✅ FastAPI server running on http://127.0.0.1:{PORT}")

# ── Expose via ngrok ─────────────────────────────────────────────────
# Your auth token (free tier: 1 concurrent tunnel, 2-hour sessions)
NGROK_AUTH_TOKEN = "2UvjyzuFvXKDBcyhyMngm8fIAox_4jyHnZnRekwUTU3VcPmUB"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

public_url = ngrok.connect(PORT)
url_str = public_url.public_url
print(f"\n{'='*60}")
print(f"🌐 PUBLIC URL: {url_str}")
print(f"{'='*60}")
print(f"\n👉 Paste this into your local .env:")
print(f"   COLAB_TTS_URL={url_str}")

In [ ]:
#@title Cell 4 — Test the endpoint (optional)
import base64
import io
import numpy as np
import requests
import soundfile as sf

# Generate a 1-second silent WAV as dummy reference audio
dummy = np.zeros(24000, dtype=np.float32)
buf = io.BytesIO()
sf.write(buf, dummy, 24000, format="WAV")
dummy_b64 = base64.b64encode(buf.getvalue()).decode()

resp = requests.post(
    "http://127.0.0.1:8899/synthesize",
    json={
        "text": "Hello world, this is a test.",
        "reference_audio_b64": dummy_b64,
        "language": "en",
    },
    timeout=120,
)
print(f"Status: {resp.status_code}")
if resp.status_code == 200:
    data = resp.json()
    print(f"Inference: {data['inference_time_s']}s")
    print(f"Audio duration: {data['duration_s']}s")
else:
    print(f"Error: {resp.json()}")